### Install Dependencies

In [0]:
!pip install kagglehub
dbutils.library.restartPython()

### Define Parameters

In [0]:
dbutils.widgets.text("CATALOG_NAME", "", "Catalog Name")
dbutils.widgets.text("SCHEMA_NAME", "", "Schema Name")
CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")

### Create Catalog, Schema, and Volume

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.kaggle_files")

### Configure Kaggle Download Path

In [0]:
import os

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/kaggle_files"
os.environ["KAGGLEHUB_CACHE"] = VOLUME_PATH

### Download Dataset from Kaggle

In [0]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shrinivasv/retail-store-star-schema-dataset")

print("Path to dataset files:", path)

### Load CSV Files into Delta Tables

In [0]:
import glob

csv_folder = path  # path variable from kagglehub.dataset_download
csv_files = glob.glob(f"{csv_folder}/*.csv")

for csv_file in csv_files:
    table_name = os.path.splitext(os.path.basename(csv_file))[0]
    df = spark.read.csv(csv_file, header=True, inferSchema=True)
    # Clean column names: replace spaces with underscores
    for c in df.columns:
        df = df.withColumnRenamed(c, c.replace(" ", "_"))
    full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"
    spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
    df.write.saveAsTable(full_table_name)